# Filters STRs

Importing libraries

In [1]:
library(rstatix)
library(stringr)
library(tidyverse)
library(gt)
library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)


Attaching package: ‘rstatix’


The following object is masked from ‘package:stats’:

    filter


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ purrr     1.1.0
✔ forcats   1.0.1     ✔ readr     2.1.5
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks rstatix::filter(), stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


**Data import**

In [2]:
# 1. Load the new dataset
df_strs <- read_tsv('../../samples/STRs_analysis_dataset.tsv')

# Rename collumns
df_filtered_grouped <- df_strs %>%
  mutate(
    sample_id = sample_id 
  )

cat("Total observations:", nrow(df_filtered_grouped), "\n")

# 2. Load the list of target genes identified by scRNA-seq analysis (COVID-19)
covid_genes_path <- "results/STR_vs_scRNA_overlap_unified.csv"
target_genes <- read_csv(covid_genes_path) %>% pull('gene_name') %>% unique()


Rows: 495844 Columns: 23
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (14): STRs_ID, group, sex, repeat_unit, gene_id, gene_name, region, chro...
dbl  (9): age, allele1_est, allele2_est, depth, start, end, n_outliers, n_cl...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Total observations: 495844 


Rows: 15 Columns: 16
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (7): gene_name, cell_type, STRs_ID, outlier_samples, GEO, source_tissue,...
dbl (9): LogFC, Pvalue, abs_res, noise_ratio, outlier_residuals, n_clusters,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


**Mann-Whitney test: case vs. control (Allele2_est)**

If there are more than 3 observations per group

In [3]:
# =======================================================
# STATISTICAL ANALYSIS: MANN-WHITNEY (ALLELE 2)
# =======================================================
cat("\n>>> Starting Mann-Whitney Analysis: Allele 2 (N > 3) <<<\n")

# 1. Data Preparation
# Ensuring target metric is defined and cleaning NAs
df_stats_prep <- df_strs %>%
  mutate(target_metric = allele2_est) %>%
  filter(!is.na(target_metric))

# 2. Filter Loci with N > 3 samples per group (Minimum 8 samples total)
eligible_loci <- df_stats_prep %>%
  group_by(STRs_ID, group) %>%
  summarise(n = n(), .groups = "drop") %>%
  pivot_wider(names_from = group, values_from = n, values_fill = 0) %>%
  # Ensure both groups meet the minimum threshold
  filter(case > 3 & control > 3) %>%
  pull(STRs_ID)

# 3. Identify Significant Loci
# We run the test to find WHICH IDs are significant under FDR correction
significant_results <- df_stats_prep %>%
  filter(STRs_ID %in% eligible_loci) %>%
  group_by(STRs_ID) %>%
  wilcox_test(target_metric ~ group) %>%
  # Apply FDR correction (Benjamini-Hochberg)
  adjust_pvalue(method = "BH") %>%
  add_significance("p.adj") %>%
  filter(p.adj < 0.05)

significant_ids <- significant_results %>% pull(STRs_ID)

# 4. Extract Full Observations for Significant Loci
# Instead of exporting a summary, we export every row associated with significant IDs
# This prevents the "Unknown" grouping issue in the plotting script
results_final <- df_strs %>%
  filter(STRs_ID %in% significant_ids) %>%
  # Merge the p-values back into the full observation set
  left_join(
    significant_results %>% select(STRs_ID,p, p.adj, p.adj.signif),
    by = "STRs_ID"
  ) %>%
  select(
    STRs_ID, gene_name, group, allele1_est, allele2_est, outlier_residuals, depth,
    p, p.adj, p.adj.signif, chrom, start, region
  ) %>%
  arrange(p.adj, STRs_ID)

# 5. Clean Output Display
cat(paste0(rep("-", 60), collapse = ""), "\n")
cat(sprintf("Loci Analyzed (N > 3):        %d\n", length(eligible_loci)))
cat(sprintf("Significant Loci (p.adj < 0.05): %d\n", length(significant_ids)))
cat(sprintf("Total Observations Exported:     %d\n", nrow(results_final)))
cat(paste0(rep("-", 60), collapse = ""), "\n\n")

if (length(significant_ids) > 0) {
  # Showing a snippet of the significant IDs found
  print(head(significant_results, 10))
} else {
  cat("No significant loci found with p.adj < 0.05 at N > 3 threshold.\n")
}

# 6. Export
if (!dir.exists("results")) dir.create("results")
output_file <- "results/allele2_mw.csv"

# write_csv2 uses ";" as delimiter and "," as decimal separator (standard for your pipeline)
write_csv2(results_final, output_file)

cat(sprintf("\n✓ Full observations for significant loci saved to: %s\n", output_file))


>>> Starting Mann-Whitney Analysis: Allele 2 (N > 3) <<<


------------------------------------------------------------ 
Loci Analyzed (N > 3):        7745
Significant Loci (p.adj < 0.05): 0
Total Observations Exported:     0
------------------------------------------------------------ 

No significant loci found with p.adj < 0.05 at N > 3 threshold.

✓ Full observations for significant loci saved to: results/allele2_mw.csv


**Mann-Whitney test: case vs. control (mean_allele)**

If there are more than 3 observations per group

In [4]:
# =======================================================
# STATISTICAL ANALYSIS: MANN-WHITNEY (MEAN ALLELE)
# =======================================================
cat("\n>>> Starting Mann-Whitney Analysis: Mean Alleles (N >= 3) <<<\n")

# 1. Data Preparation
# Calculating Mean Allele: (A1 + A2) / 2 and removing NAs
df_stats_prep <- df_strs %>%
  mutate(target_metric = (allele1_est + allele2_est) / 2) %>%
  filter(!is.na(target_metric))

# 2. Filter Loci with sufficient samples
# Requirement: n >= 3 per group for statistical robustness
eligible_loci <- df_stats_prep %>%
  group_by(STRs_ID, group) %>%
  summarise(n = n(), .groups = "drop") %>%
  pivot_wider(names_from = group, values_from = n, values_fill = 0) %>%
  filter(case >= 3 & control >= 3) %>%
  pull(STRs_ID)

# 3. Identify Significant Loci
# Perform the test to determine which STRs meet the p.adj < 0.05 threshold
significant_results <- df_stats_prep %>%
  filter(STRs_ID %in% eligible_loci) %>%
  group_by(STRs_ID) %>%
  wilcox_test(target_metric ~ group) %>%
  # Apply Benjamini-Hochberg correction
  adjust_pvalue(method = "BH") %>%
  add_significance("p.adj") %>%
  filter(p.adj < 0.05)

significant_ids <- significant_results %>% pull(STRs_ID)

# 4. Extract Full Observations for Significant Loci
# Exporting raw data points instead of a summary to support the Master Plot script
results_final <- df_strs %>%
  filter(STRs_ID %in% significant_ids) %>%
  # Merge calculated statistical significance back to the raw observations
  left_join(
    significant_results %>% select(STRs_ID, p, p.adj, p.adj.signif),
    by = "STRs_ID"
  ) %>%
  select(
    STRs_ID, gene_name, group, allele1_est, allele2_est, outlier_residuals, depth,
    p, p.adj, p.adj.signif, chrom, start, region
  ) %>%
  arrange(p.adj, STRs_ID)

# 5. Clean Output Display
cat(paste0(rep("-", 60), collapse = ""), "\n")
cat(sprintf("Loci Analyzed (N >= 3):       %d\n", length(eligible_loci)))
cat(sprintf("Significant Loci (p.adj < 0.05): %d\n", length(significant_ids)))
cat(sprintf("Total Observations Exported:     %d\n", nrow(results_final)))
cat(paste0(rep("-", 60), collapse = ""), "\n\n")

if (length(significant_ids) > 0) {
  print(head(significant_results, 10))
} else {
  cat("No significant loci found after p-value adjustment.\n")
}

# 6. Export Full Results
if (!dir.exists("results")) dir.create("results")
output_file <- "results/mean_allele_mw.csv"

# write_csv2 uses ";" as delimiter and "," as decimal separator
write_csv2(results_final, output_file)

cat(sprintf("\n✓ Full observations for significant Mean Allele loci saved to: %s\n", output_file))


>>> Starting Mann-Whitney Analysis: Mean Alleles (N >= 3) <<<


------------------------------------------------------------ 
Loci Analyzed (N >= 3):       10551
Significant Loci (p.adj < 0.05): 0
Total Observations Exported:     0
------------------------------------------------------------ 

No significant loci found after p-value adjustment.

✓ Full observations for significant Mean Allele loci saved to: results/mean_allele_mw.csv


**Identification of variants without overlap: case vs. control (mean_allele)**

In [5]:
colnames(df_strs)

[1] "STRs_ID"           "group"             "age"              
 [4] "sex"               "allele1_est"       "allele2_est"      
 [7] "depth"             "repeat_unit"       "gene_id"          
[10] "gene_name"         "region"            "chrom"            
[13] "start"             "end"               "sample_id"        
[16] "pop"               "contribution"      "type"             
[19] "n_outliers"        "outlier_samples"   "outlier_residuals"
[22] "n_clusters"        "noise_ratio"

In [6]:
# =======================================================
# ANALYSIS: SIGNATURES NO-OVERLAP (ALLELE MEAN - COVID FILTER)
# =======================================================
cat("--- Starting Targeted No-Overlap Analysis for Allele Mean (Full Discovery) ---\n")

# 1. Data Preparation and Targeted Filtering
# Metric: Arithmetic Mean (A1 + A2) / 2
df_stats_prep <- df_strs %>%
  filter(gene_name %in% target_genes) %>% 
  mutate(target_metric = (allele1_est + allele2_est) / 2) %>%
  filter(!is.na(target_metric))

cat(sprintf("Data filtered. Total observations for Allele Mean: %d\n", nrow(df_stats_prep)))

# 2. Compute Statistics per Locus and Group (For Selection Logic)
locus_stats <- df_stats_prep %>%
  group_by(STRs_ID, group) %>%
  summarise(
    median_val = median(target_metric),
    min_val = min(target_metric),
    max_val = max(target_metric),
    n_samples = n(),
    .groups = "drop"
  )

# 3. Identify No-Overlap Variants (Selection Filter)
no_overlap_ids <- locus_stats %>%
  pivot_wider(
    names_from = group, 
    values_from = c(median_val, min_val, max_val, n_samples)
  ) %>%
  # Minimum threshold to avoid single-outlier signatures
  filter(n_samples_case >= 3 & n_samples_control >= 3) %>%
  mutate(
    case_higher = min_val_case > max_val_control,
    control_higher = min_val_control > max_val_case,
    overlap_type = case_when(
      case_higher ~ "case_higher",
      control_higher ~ "control_higher",
      TRUE ~ "overlap"
    )
  ) %>%
  # Filter only for loci where [min, max] intervals do NOT cross
  filter(overlap_type != "overlap") %>%
  pull(STRs_ID)

# 4. Extract Full Observations for identified No-Overlap Loci
results_final <- df_strs %>%
  filter(STRs_ID %in% no_overlap_ids) %>%
  # Force outlier_residuals to numeric if they are currently text NAs
  mutate(mean_allele_calc = (allele1_est + allele2_est) / 2) %>%
  mutate(abs_res = abs(mean_allele_calc)) %>%  # <--- ADICIONAR
  select(
    STRs_ID, gene_name, group, mean_allele_calc, allele1_est,
    allele2_est, outlier_residuals, abs_res,  # <--- ADICIONAR abs_res
    depth, chrom, start, region
  ) %>%
  arrange(gene_name, STRs_ID)

# 5. Save and Report
if (!dir.exists("results")) dir.create("results")
output_path <- "results/covid_targeted_no_overlap_mean_allele.csv"

# write_csv2: semicolon as separator, comma as decimal
write_csv2(results_final, output_path)

cat(paste0(rep("-", 60), collapse = ""), "\n")
cat(sprintf("Loci with NO-OVERLAP identified: %d\n", length(no_overlap_ids)))
cat(sprintf("Total Observations Exported:     %d\n", nrow(results_final)))
cat(sprintf("✓ Results saved to: %s\n", output_path))
cat(paste0(rep("-", 60), collapse = ""), "\n")

if (length(no_overlap_ids) > 0) {
  cat("\n--- Preview of Exported Data (First 10 rows) ---\n")
  print(head(results_final, 10))
}

--- Starting Targeted No-Overlap Analysis for Allele Mean (Full Discovery) ---
Data filtered. Total observations for Allele Mean: 1990
------------------------------------------------------------ 
Loci with NO-OVERLAP identified: 0
Total Observations Exported:     0
✓ Results saved to: results/covid_targeted_no_overlap_mean_allele.csv
------------------------------------------------------------ 


**Identification of variants without overlap: case vs. control (allele_2est)**

In [7]:
# =======================================================
# ANALYSIS: SIGNATURES NO-OVERLAP (ALLELE 2 - COVID FILTER)
# =======================================================
cat("--- Starting Targeted No-Overlap Analysis for Allele 2 (Full Discovery) ---\n")

# Load Target Genes
if (file.exists(covid_genes_path)) {
  target_genes <- read_csv(covid_genes_path, show_col_types = FALSE) %>%
    pull(`gene_name`) %>%
    unique()
  cat(sprintf("✓ Target gene list loaded. Unique genes: %d\n", length(target_genes)))
} else {
  stop("Error: Target gene list file not found.")
}

# 1. Data Preparation and Targeted Filtering
# Metric: Specifically Allele 2 estimate
df_stats_prep <- df_strs %>%
  filter(gene_name %in% target_genes) %>% 
  mutate(target_metric = allele2_est) %>%
  filter(!is.na(target_metric))

cat(sprintf("Data filtered. Total observations for Allele 2: %d\n", nrow(df_stats_prep)))

# 2. Compute Statistics per Locus and Group (For Selection Filter)
locus_stats <- df_stats_prep %>%
  group_by(STRs_ID, group) %>%
  summarise(
    median_val = median(target_metric),
    min_val = min(target_metric),
    max_val = max(target_metric),
    n_samples = n(),
    .groups = "drop"
  )

# 3. Identify No-Overlap Variants (Selection Logic)
no_overlap_ids <- locus_stats %>%
  pivot_wider(
    names_from = group, 
    values_from = c(median_val, min_val, max_val, n_samples)
  ) %>%
  # Minimum sample size filter (N >= 3) for stability
  filter(n_samples_case >= 3 & n_samples_control >= 3) %>%
  mutate(
    case_higher = min_val_case > max_val_control,
    control_higher = min_val_control > max_val_case,
    overlap_type = case_when(
      case_higher ~ "case_higher",
      control_higher ~ "control_higher",
      TRUE ~ "overlap"
    )
  ) %>%
  # Keep only loci where Case and Control distributions are completely separate
  filter(overlap_type != "overlap") %>%
  pull(STRs_ID)

# 4. Extract Full Observations for Identified Loci
results_final <- df_strs %>%
  filter(STRs_ID %in% no_overlap_ids) %>%
  mutate(
  outlier_residuals = as.numeric(sub(";.*", "", as.character(outlier_residuals))),
  abs_res = abs(allele2_est),       # <--- ADICIONAR
  target_metric_allele2 = allele2_est
) %>%
  select(
    STRs_ID, gene_name, group, target_metric_allele2,
    allele1_est, allele2_est, outlier_residuals, abs_res,  # <--- ADICIONAR abs_res
    depth, chrom, start, region
  ) %>%
  arrange(gene_name, STRs_ID)

# --- DEBUG WITHIN STEP 4 ---
valid_resids <- sum(!is.na(results_final$outlier_residuals))
cat(sprintf("Checking identified loci: Found %d valid residuals for these %d rows.\n", 
            valid_resids, nrow(results_final)))

# 5. Save and Report
if (!dir.exists("results")) dir.create("results")
output_path <- "results/covid_targeted_no_overlap_allele2.csv"

# write_csv2: semicolon as separator, comma as decimal
write_csv2(results_final, output_path)

cat(paste0(rep("-", 60), collapse = ""), "\n")
cat(sprintf("Loci with NO-OVERLAP identified (Allele 2): %d\n", length(no_overlap_ids)))
cat(sprintf("Total Observations Exported:                 %d\n", nrow(results_final)))
cat(sprintf("✓ Results saved to: %s\n", output_path))
cat(paste0(rep("-", 60), collapse = ""), "\n")

if (length(no_overlap_ids) > 0) {
  cat("\n--- Preview of Exported Allele 2 Data ---\n")
  print(head(results_final, 10))
}

--- Starting Targeted No-Overlap Analysis for Allele 2 (Full Discovery) ---
✓ Target gene list loaded. Unique genes: 8
Data filtered. Total observations for Allele 2: 1990
Checking identified loci: Found 0 valid residuals for these 0 rows.
------------------------------------------------------------ 
Loci with NO-OVERLAP identified (Allele 2): 0
Total Observations Exported:                 0
✓ Results saved to: results/covid_targeted_no_overlap_allele2.csv
------------------------------------------------------------ 


Quality evaluation of dbscan signal

In [8]:
# =======================================================
# TECHNICAL VALIDATION: DBSCAN REFINED
# =======================================================

cat("--- Processing DBSCAN Technical Metrics & Quality Table ---\n")

# 1. Data Cleaning and Tier Definition
locus_tech_stats <- df_strs %>%
  group_by(STRs_ID, sample_id) %>%
  summarise(
    noise_ratio  = first(noise_ratio),
    n_clusters   = first(n_clusters),
    region       = first(region),
    .groups = "drop"
  ) %>%
  mutate(region = case_when(
    region == "five_prime_utr" ~ "5' UTR",
    region == "three_prime_utr" ~ "3' UTR",
    region == "intron"          ~ "Intron",
    region == "exon"            ~ "Exon",
    region == "promoter"        ~ "Promoter",
    region == "intergenic"      ~ "Intergenic",
    region == "non_coding_exons" ~"Non-coding exons",
    TRUE                        ~ region
  )) %>%
  # Quality Levels
  mutate(noise_tier = case_when(
    noise_ratio < 0.10 ~ "High Quality (noise < 0.10)",
    noise_ratio < 0.25 ~ "Acceptable (noise < 0.25)",
    TRUE                ~ "Other (> 0.25)"
  )) %>%
  # Cluster Tiers
  mutate(cluster_tier = case_when(
    n_clusters == 1 ~ "1 Cluster",
    n_clusters == 2 ~ "2 Clusters",
    n_clusters >= 3 ~ "3+ Clusters",
    TRUE            ~ "Unknown"
  )) %>%
  # Approval Criteria (Passed)
  mutate(passed_qc = noise_ratio < 0.10 & n_clusters == 1)

# 2. Generating the Summary Table (Quality Funnel)
quality_table <- locus_tech_stats %>%
  group_by(region) %>%
  summarise(
    total_observations = n(),
    # ADICIONADO: na.rm = TRUE para ignorar NAs na soma
    passed_qc_count    = sum(passed_qc, na.rm = TRUE),
    perc_passed        = (passed_qc_count / total_observations) * 100,
    avg_noise          = mean(noise_ratio, na.rm = TRUE),
    .groups = "drop"
  )

# 3. Views
cat("\n--- Generating 2-Plot Panel ---\n")

p1 <- ggplot(locus_tech_stats %>% group_by(region, cluster_tier) %>% summarise(n=n(), .groups = "drop_last") %>% mutate(p=(n/sum(n))*100), 
             aes(x = region, y = p, fill = cluster_tier)) +
  # CORRIGIDO: size alterado para linewidth para evitar o Warning do ggplot2
  geom_bar(stat = "identity", position = "fill", color = "white", linewidth = 0.2) +
  scale_y_continuous(labels = scales::percent) +
  scale_fill_manual(values = c("#A2D2FF", "#3498DB", "#2C3E50")) +
  labs(title = "A. Genotypes per Sample/Region", x = "", y = "% of Sample-Locus", fill = "Clusters") +
  theme_minimal() + theme(axis.text.x = element_text(angle = 45, hjust = 1))

p2 <- ggplot(locus_tech_stats %>% group_by(region, noise_tier) %>% summarise(n=n(), .groups = "drop_last") %>% mutate(p=(n/sum(n))*100), 
             aes(x = region, y = p, fill = noise_tier)) +
  # CORRIGIDO: size alterado para linewidth
  geom_bar(stat = "identity", position = "fill", color = "white", linewidth = 0.2) +
  scale_y_continuous(labels = scales::percent) +
  scale_fill_manual(values = c("#27ae60", "#f1c40f", "#c0392b")) +
  labs(title = "B. Noise Tiers per Sample/Region", x = "", y = "% of Sample-Locus", fill = "Quality") +
  theme_minimal() + theme(axis.text.x = element_text(angle = 45, hjust = 1))
  
# 4. Save results
final_layout <- p1 | p2
ggsave("results/dbscan_dual_panel_validation.png", final_layout, width = 14, height = 7)

# Save quality table
write_csv2(quality_table, "results/quality_funnel_summary.csv")

# Display the table in the console for a quick review
print(quality_table)

cat("\n✓ Dual plot panel saved: results/dbscan_dual_validation.png\n")
cat("✓ Quality table saved: results/quality_summary.csv\n")

--- Processing DBSCAN Technical Metrics & Quality Table ---



--- Generating 2-Plot Panel ---
# A tibble: 8 × 5
  region           total_observations passed_qc_count perc_passed avg_noise
  <chr>                         <int>           <int>       <dbl>     <dbl>
1 3' UTR                         4118            3311        80.4  0.000604
2 5' UTR                          902             742        82.3  0       
3 CDS                             158             130        82.3  0       
4 Intergenic                   196579          156251        79.5  0.000448
5 Intron                       199035          159073        79.9  0.000434
6 Non-coding exons               1356            1088        80.2  0.000919
7 Promoter                       7628            6006        78.7  0.000167
8 others                        86068           69638        80.9  0.000632

✓ Dual plot panel saved: results/dbscan_dual_validation.png
✓ Quality table saved: results/quality_summary.csv


convert to a publication ready table

In [9]:
# =======================================================
# FINAL UNIFIED REPORT: BINARY OUTLIER ANALYSIS
# =======================================================

cat("--- Generating Unified Report (0 vs >0 Outliers) ---\n")

# 1. Unified processing
df_unified_report <- bind_rows(
  # A. General Summary
  df_strs %>%
    summarise(
      total_obs = n(),
      with_signal = sum(!is.na(n_clusters) & n_clusters > 0),
      no_signal = total_obs - with_signal
    ) %>%
    pivot_longer(cols = everything(), names_to = "metric", values_to = "raw_value") %>%
    mutate(
      category = "General Summary",
      percentage = round((raw_value / first(raw_value)) * 100, 2)
    ),
  
  # B. Cluster Distribution (Signal observations only)
  df_strs %>%
    filter(!is.na(n_clusters) & n_clusters > 0) %>%
    group_by(n_clusters) %>%
    summarise(raw_value = n(), .groups = "drop") %>%
    mutate(
      category = "Cluster Distribution",
      metric = paste0(n_clusters, " Cluster(s)"),
      percentage = round((raw_value / sum(raw_value)) * 100, 2)
    ) %>%
    select(-n_clusters),
  
  # C. Noise Tiers (Signal observations only)
  df_strs %>%
    filter(!is.na(n_clusters) & n_clusters > 0) %>%
    mutate(metric = case_when(
      noise_ratio <= 0.05 ~ "Noise 00-05%",
      noise_ratio <= 0.10 ~ "Noise 05-10%",
      noise_ratio <= 0.20 ~ "Noise 10-20%",
      noise_ratio <= 0.50 ~ "Noise 20-50%",
      TRUE                ~ "Noise > 50%"
    )) %>%
    group_by(metric) %>%
    summarise(raw_value = n(), .groups = "drop") %>%
    mutate(
      category = "Noise Tiers",
      percentage = round((raw_value / sum(raw_value)) * 100, 2)
    ),

  # D. Outlier Comparison (0 vs >0)
  df_strs %>%
    filter(!is.na(n_clusters) & n_clusters > 0) %>%
    mutate(has_outlier = ifelse(n_outliers > 0, "Observations with >0 Outliers", "Observations with 0 Outliers")) %>%
    group_by(has_outlier) %>%
    summarise(raw_value = n(), .groups = "drop") %>%
    mutate(
      category = "Outlier Frequency",
      percentage = round((raw_value / sum(raw_value)) * 100, 2)
    ) %>%
    rename(metric = has_outlier)
) %>%
  select(category, metric, raw_value, percentage)

# 2. Export
write_csv2(df_unified_report, "results/unified_binary_outlier_report.csv")

# Display
print(df_unified_report, n = Inf)

cat("\n✓ Binary outlier report saved successfully!\n")

--- Generating Unified Report (0 vs >0 Outliers) ---


# A tibble: 8 × 4
  category             metric                        raw_value percentage
  <chr>                <chr>                             <int>      <dbl>
1 General Summary      total_obs                        495844     100   
2 General Summary      with_signal                      396239      79.9 
3 General Summary      no_signal                         99605      20.1 
4 Cluster Distribution 1 Cluster(s)                     396239     100   
5 Noise Tiers          Noise 00-05%                     395784      99.9 
6 Noise Tiers          Noise 05-10%                        455       0.11
7 Outlier Frequency    Observations with 0 Outliers     389098      98.2 
8 Outlier Frequency    Observations with >0 Outliers      7141       1.8 

✓ Binary outlier report saved successfully!


In [10]:
# =======================================================
# FINAL UNIFIED REPORT: ANALYTICS & PUBLICATION TABLE
# =======================================================

library(tidyverse)
library(gt)

cat("--- Generating Unified English Report & Publication Table ---\n")

# 1. Unified Data Processing
df_unified_report <- bind_rows(
  # A. General Summary
  df_strs %>%
    summarise(
      total_obs = n(),
      with_signal = sum(!is.na(n_clusters) & n_clusters > 0),
      no_signal = total_obs - with_signal
    ) %>%
    pivot_longer(cols = everything(), names_to = "metric", values_to = "raw_value") %>%
    mutate(
      category = "General Summary",
      percentage = round((raw_value / first(raw_value)) * 100, 2)
    ),
  
  # B. Cluster Distribution (Signal observations only)
  df_strs %>%
    filter(!is.na(n_clusters) & n_clusters > 0) %>%
    group_by(n_clusters) %>%
    summarise(raw_value = n(), .groups = "drop") %>%
    mutate(
      category = "Cluster Distribution",
      metric = paste0(n_clusters, " Cluster(s)"),
      percentage = round((raw_value / sum(raw_value)) * 100, 2)
    ) %>%
    select(-n_clusters),
  
  # C. Noise Tiers (Signal observations only)
  df_strs %>%
    filter(!is.na(n_clusters) & n_clusters > 0) %>%
    mutate(metric = case_when(
      noise_ratio <= 0.05 ~ "Noise 00-05%",
      noise_ratio <= 0.10 ~ "Noise 05-10%",
      noise_ratio <= 0.20 ~ "Noise 10-20%",
      noise_ratio <= 0.50 ~ "Noise 20-50%",
      TRUE                ~ "Noise > 50%"
    )) %>%
    group_by(metric) %>%
    summarise(raw_value = n(), .groups = "drop") %>%
    mutate(
      category = "Noise Tiers",
      percentage = round((raw_value / sum(raw_value)) * 100, 2)
    ),

  # D. Outlier Comparison (Signal observations only)
  df_strs %>%
    filter(!is.na(n_clusters) & n_clusters > 0) %>%
    mutate(has_outlier = ifelse(n_outliers > 0, "Observations with >0 Outliers", "Observations with 0 Outliers")) %>%
    group_by(has_outlier) %>%
    summarise(raw_value = n(), .groups = "drop") %>%
    mutate(
      category = "Outlier Frequency",
      percentage = round((raw_value / sum(raw_value)) * 100, 2)
    ) %>%
    rename(metric = has_outlier)
) %>%
  select(category, metric, raw_value, percentage)

# 2. Export to CSV
write_csv2(df_unified_report, "results/unified_technical_report.csv")
cat("✓ CSV exported to results/unified_technical_report.csv\n")

# 3. Rename metrics for friendly publication labels
df_friendly <- df_unified_report %>%
  mutate(metric = case_when(
    metric == "total_obs"     ~ "Total Observations",
    metric == "with_signal"   ~ "Observations with Signal",
    metric == "no_signal"     ~ "Observations without Signal",
    metric == "1 Cluster(s)"  ~ "1 Cluster",
    TRUE ~ metric
  ))

# 4. Generate Professional Table (gt)
publication_table <- df_friendly %>%
  group_by(category) %>%
  gt() %>%
  tab_header(
    title = md("**Table 1. Technical Validation of STR Genotyping**"),
    subtitle = "DBSCAN efficiency, cluster stability, and noise metrics"
  ) %>%
  cols_label(
    metric = "Metric",
    raw_value = "Absolute Frequency (n)",
    percentage = "Relative Frequency (%)"
  ) %>%
  # Formatting numbers
  fmt_number(
    columns = c(raw_value),
    decimals = 0,
    use_seps = TRUE
  ) %>%
  fmt_number(
    columns = c(percentage),
    decimals = 2
  ) %>%
  # Add % suffix
  text_transform(
    locations = cells_body(columns = c(percentage)),
    fn = function(x) paste0(x, "%")
  ) %>%
  # Professional Styling
  tab_options(
    table.width = px(750),
    column_labels.font.weight = "bold",
    row_group.font.weight = "bold",
    row_group.background.color = "#F9F9F9",
    table.font.size = px(14),
    table.border.top.color = "black",
    table.border.bottom.color = "black",
    heading.align = "left",
    data_row.padding = px(6)
  ) %>%
  cols_align(align = "left", columns = c(metric)) %>%
  cols_align(align = "center", columns = c(raw_value, percentage)) %>%
  tab_source_note(
    source_note = md("*Note: Signal detection and noise metrics are calculated based on DBSCAN clustering parameters.*")
  )

# 5. Save Table Files
gtsave(publication_table, "results/Technical_Validation_Table.html")
cat("✓ Publication-ready table saved as HTML: results/Technical_Validation_Table.html\n")


--- Generating Unified English Report & Publication Table ---


✓ CSV exported to results/unified_technical_report.csv
✓ Publication-ready table saved as HTML: results/Technical_Validation_Table.html


TOP 10 UNIQUE LOCI OUTLIERS

In [11]:
# =======================================================
# EXTRACTION: TOP 10 UNIQUE LOCI OUTLIERS (EXPANDED)
# =======================================================
cat("--- Extracting the top outlier for 10 unique genes (Full Observation Format) ---\n")

# 1. Process and Identify the Top 10 Outlier Events
# We expand the outliers to find exactly which samples and residuals are the highest
top_outlier_events <- df_strs %>%
  # Quality Filters
  filter(n_clusters > 0) %>%
  filter(noise_ratio <= 0.10) %>%
  filter(n_outliers > 0) %>%
  
  # Ensure columns are character for separation
  mutate(
    outlier_samples = as.character(outlier_samples),
    outlier_residuals = as.character(outlier_residuals)
  ) %>%
  
  # Expand rows so each outlier sample has its own row
  separate_rows(outlier_samples, outlier_residuals, sep = ";") %>%
  mutate(
    abs_res = abs(as.numeric(outlier_residuals))
  ) %>%
  filter(!is.na(abs_res)) %>%
  
  # Select the single highest residual per STR ID
  group_by(STRs_ID) %>%
  slice_max(abs_res, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  
  # Rank globally and take the top 10 unique loci
  arrange(desc(abs_res)) %>%
  slice(1:10)

# 2. Extract Full Observations for these 10 Loci
# To avoid "Unknown" columns in the plot, we retrieve all samples for these 10 STRs
# from the original dataframe, ensuring 'group' labels are present.
significant_ids <- top_outlier_events %>% pull(STRs_ID)

results_final <- df_strs %>%
  filter(STRs_ID %in% significant_ids) %>%
  # We manually attach the 'abs_res' calculated above so the bubble plot 
  # knows which intensity to use for these specific loci
  left_join(
    top_outlier_events %>% select(STRs_ID, abs_res),
    by = "STRs_ID"
  ) %>%
  select(
    STRs_ID, gene_name, group, allele1_est, allele2_est, outlier_residuals,
    abs_res, # This will drive the 'intensity' in your Master Function
    any_of(c("depth")),
    chrom, start, region
  ) %>%
  arrange(desc(abs_res))

# =======================================================
# OUTPUT & EXPORT
# =======================================================

cat("\n=== TOP 10 UNIQUE LOCI (Ranked by Outlier Magnitude) ===\n")
# Show the IDs and their associated genes
print(results_final %>% select(gene_name, STRs_ID, abs_res) %>% distinct())

if (!dir.exists("results")) dir.create("results")

output_file <- "results/top_10_unique_loci.csv"

# write_csv2: ";" delimiter, "," decimal separator
write_csv2(results_final, output_file)

cat(sprintf("\n✓ Results saved to: %s\n", output_file))
cat(sprintf("Total rows exported (Full group context): %d\n", nrow(results_final)))

--- Extracting the top outlier for 10 unique genes (Full Observation Format) ---



=== TOP 10 UNIQUE LOCI (Ranked by Outlier Magnitude) ===
# A tibble: 10 × 3
   gene_name STRs_ID               abs_res
   <chr>     <chr>                   <dbl>
 1 HMGCLL1   chr6:55543354:AT:20     157. 
 2 .         chr3:171524287:CTG:11   130. 
 3 CDK19     chr6:110734520:CT:21    118. 
 4 ITFG2-AS1 chr12:2720571:AC:287    112. 
 5 .         chr3:194324653:AC:10     89.1
 6 TAFA4     chr3:68873337:AC:20      84.9
 7 EFL1      chr15:82211595:AC:12     83.4
 8 MACC1     chr7:20140802:AC:50      80.4
 9 .         chr11:115523088:AG:21    76.2
10 LINC01122 chr2:58585933:AT:70      76.0

✓ Results saved to: results/top_10_unique_loci.csv
Total rows exported (Full group context): 434
